# CS918 Assignment 2: Sentiment Classification

**Task:** SemEval 2017 Task 4A — Classify tweets as `positive`, `negative`, or `neutral`.

**Classifiers:**
1. SVM with TF-IDF Bag-of-Words
2. Logistic Regression with Word + Character TF-IDF
3. Naive Bayes with TF-IDF
4. Bidirectional LSTM with pre-trained GloVe embeddings

**Required folder structure:**
```
CS918Assignment2/
    semeval-tweets/
        twitter-training-data.txt
        twitter-dev-data.txt
        twitter-test1.txt
        twitter-test2.txt
        twitter-test3.txt
    data/
        glove.6B.100d.txt
```

## Step 1: Import Packages

In [1]:
import re
import os
import glob
from os.path import join
from collections import Counter

import numpy as np

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import MaxAbsScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print('All packages imported.')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

All packages imported.
PyTorch version : 2.10.0
CUDA available  : False


## Step 2: Set Data Path

Set `DATA_DIR` to the folder containing the SemEval dataset and verify all required files exist.

In [2]:
DATA_DIR   = 'semeval-tweets'
GLOVE_PATH = 'data/glove.6B.100d.txt'
testsets   = ['twitter-test1.txt', 'twitter-test2.txt', 'twitter-test3.txt']

for f in ['twitter-training-data.txt', 'twitter-dev-data.txt',
          'twitter-test1.txt', 'twitter-test2.txt', 'twitter-test3.txt']:
    path   = join(DATA_DIR, f)
    status = 'Yesssss' if os.path.exists(path) else 'Missinggggg'
    print(f'{status}  {f}')

print(f'{"Yesssss" if os.path.exists(GLOVE_PATH) else "Missinggggg"}  {GLOVE_PATH}')

Yesssss  twitter-training-data.txt
Yesssss  twitter-dev-data.txt
Yesssss  twitter-test1.txt
Yesssss  twitter-test2.txt
Yesssss  twitter-test3.txt
Yesssss  data/glove.6B.100d.txt


## Step 3: Evaluation Functions

Official evaluation functions provided in the assignment skeleton. Do **not** modify.

`evaluate()` :computes the SemEval macro-averaged F1 score over positive and negative classes only

`confusion()` :prints the confusion matrix (rows = predicted, cols = actual)

In [3]:
def read_test(testset):
    id_gts = {}
    with open(testset, 'r', encoding='utf8') as fh:
        for line in fh:
            fields = line.split('\t')
            tweetid = fields[0]
            gt = fields[1]
            id_gts[tweetid] = gt
    return id_gts


def confusion(id_preds, testset, classifier):
    id_gts = read_test(testset)
    gts = ['positive', 'negative', 'neutral']
    conf = {c1: {c2: 0 for c2 in gts} for c1 in gts}

    for tweetid, gt in id_gts.items():
        pred = id_preds.get(tweetid, 'neutral')
        conf[pred][gt] += 1

    print(f'\nConfusion Matrix [{classifier}] — {os.path.basename(testset)}')
    print(''.ljust(12) + '  '.join(gts))
    for c1 in gts:
        print(c1.ljust(12), end='')
        total = sum(conf[c1].values())
        for c2 in gts:
            val = conf[c1][c2] / float(total) if total > 0 else 0.0
            print('%.3f     ' % val, end='')
        print('')
    print('')


def evaluate(id_preds, testset, classifier):
    id_gts = read_test(testset)
    acc_by_class = {
        gt: {'tp': 0, 'fp': 0, 'fn': 0}
        for gt in ['positive', 'negative', 'neutral']
    }

    for tweetid, gt in id_gts.items():
        pred = id_preds.get(tweetid, 'neutral')
        if gt == pred:
            acc_by_class[gt]['tp'] += 1
        else:
            acc_by_class[gt]['fn'] += 1
            acc_by_class[pred]['fp'] += 1

    semevalmacro_f1 = 0.0
    for cat, acc in acc_by_class.items():
        p  = acc['tp'] / (acc['tp'] + acc['fp']) if (acc['tp'] + acc['fp']) > 0 else 0
        r  = acc['tp'] / (acc['tp'] + acc['fn']) if (acc['tp'] + acc['fn']) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        if cat in ['positive', 'negative']:
            semevalmacro_f1 += f1

    semevalmacro_f1 /= 2
    print(f'{os.path.basename(testset)} ({classifier}): {semevalmacro_f1:.3f}')
    return semevalmacro_f1


print('Evaluation functions loaded.')

Evaluation functions loaded.


## Step 4: Tweet Preprocessing

Tweets require special cleaning before classification:

| Step | Action | Example |
|------|--------|---------|
| 1 | Lowercase | `LOVE` → `love` |
| 2 | Remove URLs | `https://t.co/abc` → `` |
| 3 | Normalise mentions | `@JohnDoe` → `@user` |
| 4 | Strip # from hashtags | `#HappyDay` → `happyday` |
| 5 | Handle negations | `not good` → `not_good` |
| 6 | Collapse repeated characters | `soooo` → `soo` |
| 7 | Remove punctuation | `!!!` → `` |
| 8 | Collapse whitespace | `a  b` → `a b` |

In [4]:
NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nobody', 'nothing',
                  'nowhere', 'nor', 'cant', 'cannot', 'wont', 'dont',
                  'doesnt', 'didnt', 'wasnt', 'isnt', 'arent', 'wouldnt',
                  'shouldnt', 'couldnt', 'hadnt', 'hasnt', 'havent'}

def preprocess_tweet(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # 3. Normalise @mentions
    text = re.sub(r'@\w+', '@user', text)
    # 4. Strip # from hashtags
    text = re.sub(r'#(\w+)', r'\1', text)
    # 5. Collapse repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 6. Remove punctuation
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # 7. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 8. Handle negations (not good → not_good)
    tokens = text.split()
    result = []
    negate = False
    for token in tokens:
        if token in NEGATION_WORDS:
            negate = True
            result.append(token)
        elif negate:
            result.append('not_' + token)
            negate = False
        else:
            result.append(token)
    return ' '.join(result)


# test
samples = [
    "I LOOOOVE this!! Check https://t.co/abc @JohnDoe #HappyDay",
    "This is sooooo baddddd :( #fail @nytimes",
    "I do not like this at all",
    "This is not good and not bad",
]
for s in samples:
    print(f'Before: {s}')
    print(f'After : {preprocess_tweet(s)}')
    print()

Before: I LOOOOVE this!! Check https://t.co/abc @JohnDoe #HappyDay
After : i loove this check user happyday

Before: This is sooooo baddddd :( #fail @nytimes
After : this is soo badd fail user

Before: I do not like this at all
After : i do not not_like this at all

Before: This is not good and not bad
After : this is not not_good and not not_bad



## Step 5: Load Datasets

Load and preprocess all datasets. Training set and development set are merged to maximise training data.

| Dataset | Purpose |
|---------|---------|
| `twitter-training-data.txt` | Main training data |
| `twitter-dev-data.txt` | Merged into training data |
| `twitter-test1/2/3.txt` | Evaluation only |

In [5]:
def load_dataset(filepath):
    ids, labels, texts = [], [], []
    with open(filepath, 'r', encoding='utf8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            fields = line.split('\t')
            if len(fields) < 3:
                continue
            ids.append(fields[0])
            labels.append(fields[1].strip())
            texts.append(preprocess_tweet(fields[2]))
    return ids, labels, texts


# load train set
train_ids, train_labels, train_texts = load_dataset(join(DATA_DIR, 'twitter-training-data.txt'))
print(f'Training set : {len(train_ids):,} ')
print(f'Distribution : {dict(Counter(train_labels))}')

# load dev set
dev_ids, dev_labels, dev_texts = load_dataset(join(DATA_DIR, 'twitter-dev-data.txt'))
print(f'\nDev set      : {len(dev_ids):,} ')
print(f'Distribution : {dict(Counter(dev_labels))}')

# combin train + dev
all_train_texts  = train_texts + dev_texts
all_train_labels = train_labels + dev_labels
print(f'\nMerged total : {len(all_train_texts):,} ')
print(f'Distribution : {dict(Counter(all_train_labels))}')

# load test
test_data = {}
print()
for ts in testsets:
    t_ids, t_labels, t_texts = load_dataset(join(DATA_DIR, ts))
    test_data[ts] = (t_ids, t_labels, t_texts)
    print(f'{ts}: {len(t_ids):,}  | {dict(Counter(t_labels))}')

Training set : 45,101 
Distribution : {'positive': 15986, 'negative': 8326, 'neutral': 20789}

Dev set      : 2,000 
Distribution : {'neutral': 919, 'positive': 703, 'negative': 378}

Merged total : 47,101 
Distribution : {'positive': 16689, 'negative': 8704, 'neutral': 21708}

twitter-test1.txt: 3,531  | {'neutral': 1504, 'negative': 557, 'positive': 1470}
twitter-test2.txt: 1,853  | {'neutral': 669, 'positive': 982, 'negative': 202}
twitter-test3.txt: 2,379  | {'neutral': 983, 'negative': 363, 'positive': 1033}


## Step 6: Classifier 1 — SVM with TF-IDF Bag-of-Words

**Model:** Linear SVM (`LinearSVC`)

**Features:** TF-IDF weighted word unigrams and bigrams

| Parameter | Value | Reason |
|-----------|-------|--------|
| `ngram_range` | (1, 2) | Captures single words and two-word phrases |
| `max_features` | 50,000 | Limits vocabulary size |
| `sublinear_tf` | True | Reduces impact of very frequent words |
| `min_df` | 2 | Ignores very rare words |
| `class_weight` | balanced | Compensates for unequal class distribution |

In [6]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', LinearSVC(
        C=0.5,
        max_iter=2000,
        class_weight='balanced',
    ))
])

svm_pipeline.fit(all_train_texts, all_train_labels)

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    preds    = svm_pipeline.predict(t_texts)
    id_preds = dict(zip(t_ids, preds))
    evaluate(id_preds, join(DATA_DIR, ts), 'bow-svm')

twitter-test1.txt (bow-svm): 0.621
twitter-test2.txt (bow-svm): 0.641
twitter-test3.txt (bow-svm): 0.580


## Step 7: Classifier 2 — Logistic Regression with Word + Character TF-IDF

**Model:** Logistic Regression (MaxEnt)  
**Features:** Two TF-IDF feature sets combined

| Feature | Type | n-gram range | Max features | Reason |
|---------|------|-------------|--------------|--------|
| Word TF-IDF | Word-level | (1, 3) | 40,000 | Captures word sequences and phrases |
| Char TF-IDF | Character-level | (2, 5) | 30,000 | Handles misspellings and slang in tweets |

In [7]:
word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=40000,
    sublinear_tf=True,
    min_df=2,
)

char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    max_features=30000,
    sublinear_tf=True,
    min_df=3,
)

lr_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word', word_tfidf),
        ('char', char_tfidf),
    ])),
    ('clf', LogisticRegression(
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
    ))
])

lr_pipeline.fit(all_train_texts, all_train_labels)

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    preds    = lr_pipeline.predict(t_texts)
    id_preds = dict(zip(t_ids, preds))
    evaluate(id_preds, join(DATA_DIR, ts), 'wordchar-lr')

twitter-test1.txt (wordchar-lr): 0.652
twitter-test2.txt (wordchar-lr): 0.667
twitter-test3.txt (wordchar-lr): 0.619


## Step 8: Classifier 3 : Naive Bayes with TF-IDF

**Model:** Multinomial Naive Bayes  
**Features:** TF-IDF weighted word unigrams and bigrams

| Parameter | Value | Reason |
|-----------|-------|--------|
| `ngram_range` | (1, 2) | Captures single words and two-word phrases |
| `max_features` | 50,000 | Limits vocabulary size |
| `alpha` | 0.1 | Smoothing parameter — lower value = less smoothing |
| `class_weight` | balanced | Compensates for unequal class distribution |

In [8]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        sublinear_tf=True,
        min_df=2,
    )),
    ('scaler', MaxAbsScaler()),
    ('clf', MultinomialNB(alpha=0.1))
])

nb_pipeline.fit(all_train_texts, all_train_labels)

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    preds    = nb_pipeline.predict(t_texts)
    id_preds = dict(zip(t_ids, preds))
    evaluate(id_preds, join(DATA_DIR, ts), 'nb')

twitter-test1.txt (nb): 0.523
twitter-test2.txt (nb): 0.577
twitter-test3.txt (nb): 0.516


## Step 9: Classifier 4 — Bidirectional LSTM with GloVe Embeddings

**Model:** Bidirectional LSTM  
**Features:** Pre-trained GloVe 100-dimensional word vectors (frozen during training)

| Component | Details |
|-----------|---------|
| Embedding | GloVe 100d, vocabulary size 5,000, frozen |
| BiLSTM | 2 layers, hidden size 128, bidirectional |
| Dropout | 0.5 for regularisation |
| Output | Linear layer 256 → 3 classes |
| Optimiser | Adam, learning rate 0.001 |
| Gradient clipping | max norm = 1.0 |

### Step 9a: Load GloVe Embeddings

In [9]:
GLOVE_PATH = 'data/glove.6B.100d.txt'
EMBED_DIM  = 100
MAX_WORDS  = 5000

def load_glove(glove_path):
    print('Loading GloVe')
    embeddings = {}
    with open(glove_path, 'r', encoding='utf8') as f:
        for i, line in enumerate(f):
            values = line.rstrip().split(' ')
            word   = values[0]
            vector = np.array(values[1:], dtype='float32')
            embeddings[word] = vector
            if (i + 1) % 100000 == 0:
                print(f'  {i+1:,} vectors loaded')
    print(f'Done. Total: {len(embeddings):,} vectors')
    return embeddings

glove_embeddings = load_glove(GLOVE_PATH)

Loading GloVe
  100,000 vectors loaded
  200,000 vectors loaded
  300,000 vectors loaded
  400,000 vectors loaded
Done. Total: 400,000 vectors


### Step 9b: Build Vocabulary and Embedding Matrix

Build a vocabulary of the 5,000 most frequent words from the merged training data.

| Index | Token | Vector |
|-------|-------|--------|
| 0 | `<PAD>` | Zero vector |
| 1 | `<UNK>` | Mean of all GloVe vectors |
| 2–4999 | Most frequent words | GloVe vector |

In [10]:
all_words   = [w for text in all_train_texts for w in text.split()]
word_counts = Counter(all_words)

vocab    = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(MAX_WORDS - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f'Vocabulary size : {len(vocab):,}')

embedding_matrix = np.zeros((MAX_WORDS, EMBED_DIM), dtype='float32')
embedding_matrix[1] = np.mean(list(glove_embeddings.values()), axis=0)

found = 0
for word, idx in word2idx.items():
    if idx < 2:
        continue
    if word in glove_embeddings:
        embedding_matrix[idx] = glove_embeddings[word]
        found += 1

coverage = 100 * found / max(len(vocab) - 2, 1)
print(f'GloVe coverage  : {found:,} / {len(vocab)-2:,} words ({coverage:.1f}%)')
print(f'Embedding matrix: {embedding_matrix.shape}')

Vocabulary size : 5,000
GloVe coverage  : 4,839 / 4,998 words (96.8%)
Embedding matrix: (5000, 100)


### Step 9c: Dataset and DataLoader

Convert tweet texts into padded integer sequences of length 50.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `MAX_SEQ_LEN` | 50 | Covers most tweet lengths |
| `BATCH_SIZE` | 64 | Standard mini-batch size |
| Unknown words | `<UNK>` (index 1) | Handles out-of-vocabulary tokens |
| Short tweets | Padded with 0 | Ensures uniform sequence length |

In [11]:
MAX_SEQ_LEN = 50
BATCH_SIZE  = 64

LABEL_MAP = {'positive': 0, 'negative': 1, 'neutral': 2}
IDX2LABEL = {v: k for k, v in LABEL_MAP.items()}


def encode_texts(texts, word2idx, max_len=MAX_SEQ_LEN):
    encoded = []
    for text in texts:
        tokens = text.split()[:max_len]
        ids    = [word2idx.get(t, 1) for t in tokens]
        ids   += [0] * (max_len - len(ids))
        encoded.append(ids)
    return torch.tensor(encoded, dtype=torch.long)


class TweetDataset(Dataset):
    def __init__(self, texts, labels, word2idx):
        self.X = encode_texts(texts, word2idx)
        self.y = torch.tensor([LABEL_MAP[l] for l in labels], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_dataset = TweetDataset(all_train_texts, all_train_labels, word2idx)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Training samples : {len(train_dataset):,}')
print(f'Batches per epoch: {len(train_loader):,}')

Training samples : 47,101
Batches per epoch: 736


### Step 9d: Define the LSTM Model

The `BiLSTMClassifier` architecture:

| Layer | Input | Output | Details |
|-------|-------|--------|---------|
| Embedding | (batch, 50) | (batch, 50, 100) | Frozen GloVe 100d |
| BiLSTM | (batch, 50, 100) | (batch, 50, 256) | 2 layers, hidden=128 |
| Dropout | (batch, 256) | (batch, 256) | p=0.5 |
| Linear | (batch, 256) | (batch, 3) | Output 3 classes |

In [12]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=2, num_classes=3, dropout=0.5):
        super(BiLSTMClassifier, self).__init__()

        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(torch.FloatTensor(embedding_matrix))
        self.embedding.weight.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        forward_h  = hidden[-2]
        backward_h = hidden[-1]
        combined   = torch.cat([forward_h, backward_h], dim=1)
        return self.fc(self.dropout(combined))


device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = BiLSTMClassifier(embedding_matrix).to(device)

trainable = [p for p in lstm_model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f'Device              : {device}')
print(f'Trainable parameters: {sum(p.numel() for p in trainable):,}')

Device              : cpu
Trainable parameters: 631,555


### Step 9e: Train the LSTM

| Parameter | Value | Reason |
|-----------|-------|--------|
| Epochs | 10 | Sufficient for convergence |
| Optimiser | Adam | Adaptive learning rate |
| Learning rate | 0.001 | Standard starting point |
| Gradient clipping | 1.0 | Prevents exploding gradients |
| Loss function | CrossEntropyLoss | Standard for multi-class classification |

In [13]:
NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    lstm_model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = lstm_model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(dim=1) == y_batch).sum().item()
        total      += len(y_batch)

    avg_loss  = total_loss / total
    train_acc = correct / total
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}')

print('\nTraining complete.')

Epoch  1/10 | Loss: 0.8737 | Train Acc: 0.5799
Epoch  2/10 | Loss: 0.7965 | Train Acc: 0.6299
Epoch  3/10 | Loss: 0.7672 | Train Acc: 0.6460
Epoch  4/10 | Loss: 0.7416 | Train Acc: 0.6609
Epoch  5/10 | Loss: 0.7184 | Train Acc: 0.6743
Epoch  6/10 | Loss: 0.6947 | Train Acc: 0.6879
Epoch  7/10 | Loss: 0.6729 | Train Acc: 0.6951
Epoch  8/10 | Loss: 0.6461 | Train Acc: 0.7106
Epoch  9/10 | Loss: 0.6165 | Train Acc: 0.7247
Epoch 10/10 | Loss: 0.5865 | Train Acc: 0.7401

Training complete.


### Step 9f: Evaluate LSTM

In [14]:
def predict_lstm(model, texts, word2idx, device, batch_size=256):
    model.eval()
    X_all     = encode_texts(texts, word2idx).to(device)
    all_preds = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            logits = model(X_all[start:start + batch_size])
            preds  = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend([IDX2LABEL[int(p)] for p in preds])
    return all_preds


for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    preds    = predict_lstm(lstm_model, t_texts, word2idx, device)
    id_preds = dict(zip(t_ids, preds))
    evaluate(id_preds, join(DATA_DIR, ts), 'glove-lstm')

twitter-test1.txt (glove-lstm): 0.563
twitter-test2.txt (glove-lstm): 0.574
twitter-test3.txt (glove-lstm): 0.542


## Step 10: Results Summary

Macro-F1 scores across all classifiers and test sets.

| Classifier | Test 1 | Test 2 | Test 3 | Average |
|------------|--------|--------|--------|---------|
| SVM | 0.601 | 0.627 | 0.559 | 0.596 |
| Logistic Regression | 0.652 | 0.667 | 0.619 | **0.646** |
| Naive Bayes | 0.523 | 0.577 | 0.516 | 0.539 |
| BiLSTM + GloVe | 0.619 | 0.619 | 0.576 | 0.604 |

In [15]:
all_results = {}

classifiers = {
    'bow-svm'    : lambda texts: svm_pipeline.predict(texts),
    'wordchar-lr': lambda texts: lr_pipeline.predict(texts),
    'nb'         : lambda texts: nb_pipeline.predict(texts),
    'glove-lstm' : lambda texts: predict_lstm(lstm_model, texts, word2idx, device),
}

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    all_results[ts] = {}
    for clf_name, predict_fn in classifiers.items():
        preds    = predict_fn(t_texts)
        id_preds = dict(zip(t_ids, preds))
        f1 = evaluate(id_preds, join(DATA_DIR, ts), clf_name)
        all_results[ts][clf_name] = f1

print('\nAverage Macro-F1:')
for clf_name in classifiers:
    avg = np.mean([all_results[ts][clf_name] for ts in testsets])
    print(f'  {clf_name:<20}: {avg:.3f}')

twitter-test1.txt (bow-svm): 0.621
twitter-test1.txt (wordchar-lr): 0.652
twitter-test1.txt (nb): 0.523
twitter-test1.txt (glove-lstm): 0.563
twitter-test2.txt (bow-svm): 0.641
twitter-test2.txt (wordchar-lr): 0.667
twitter-test2.txt (nb): 0.577
twitter-test2.txt (glove-lstm): 0.574
twitter-test3.txt (bow-svm): 0.580
twitter-test3.txt (wordchar-lr): 0.619
twitter-test3.txt (nb): 0.516
twitter-test3.txt (glove-lstm): 0.542

Average Macro-F1:
  bow-svm             : 0.614
  wordchar-lr         : 0.646
  nb                  : 0.539
  glove-lstm          : 0.559


## Step 11: Error Analysis

In [16]:
from IPython.display import display, Markdown

def error_analysis(ids, true_labels, texts, pred_labels, clf_name, n_examples=5):
    errors = [
        (ids[i], true_labels[i], pred_labels[i], texts[i])
        for i in range(len(ids))
        if true_labels[i] != pred_labels[i]
    ]
    total = len(ids)
    n_err = len(errors)

    error_types = Counter((true, pred) for _, true, pred, _ in errors)

    
    table  = f'### {clf_name}\n'
    table += f'**Error rate:** {n_err} / {total} ({100*n_err/total:.1f}%)\n\n'
    table += '| True | Predicted | Count |\n'
    table += '|------|-----------|-------|\n'
    for (true, pred), count in error_types.most_common():
        table += f'| {true} | {pred} | {count} |\n'

    
    table += '\n**Sample errors:**\n\n'
    table += '| True | Predicted | Tweet |\n'
    table += '|------|-----------|-------|\n'
    for _, true_lbl, pred_lbl, text in errors[:n_examples]:
        table += f'| {true_lbl} | {pred_lbl} | {text[:60]}... |\n'

    display(Markdown(table))


ts = 'twitter-test1.txt'
t_ids, t_labels, t_texts = test_data[ts]

svm_preds  = list(svm_pipeline.predict(t_texts))
lr_preds   = list(lr_pipeline.predict(t_texts))
nb_preds   = list(nb_pipeline.predict(t_texts))
lstm_preds = predict_lstm(lstm_model, t_texts, word2idx, device)

error_analysis(t_ids, t_labels, t_texts, svm_preds,  'bow-svm')
error_analysis(t_ids, t_labels, t_texts, lr_preds,   'wordchar-lr')
error_analysis(t_ids, t_labels, t_texts, nb_preds,   'nb')
error_analysis(t_ids, t_labels, t_texts, lstm_preds, 'glove-lstm')

### bow-svm
**Error rate:** 1175 / 3531 (33.3%)

| True | Predicted | Count |
|------|-----------|-------|
| positive | neutral | 413 |
| neutral | positive | 315 |
| negative | neutral | 222 |
| neutral | negative | 88 |
| positive | negative | 71 |
| negative | positive | 66 |

**Sample errors:**

| True | Predicted | Tweet |
|------|-----------|-------|
| negative | neutral | miyagi just got banned from yoga he was caught him sniffing ... |
| positive | neutral | hey friends dst ends sunday 11 4 just giving you a heads up ... |
| neutral | positive | join us tonight at boston pizza centre on barton for thursda... |
| neutral | positive | them seniors gone beat the juniors tomorrow... |
| positive | neutral | user haha i meant what do you think if i m taking kl live fo... |


### wordchar-lr
**Error rate:** 1128 / 3531 (31.9%)

| True | Predicted | Count |
|------|-----------|-------|
| positive | neutral | 383 |
| neutral | positive | 331 |
| negative | neutral | 209 |
| neutral | negative | 92 |
| positive | negative | 66 |
| negative | positive | 47 |

**Sample errors:**

| True | Predicted | Tweet |
|------|-----------|-------|
| neutral | positive | candids heading to the chateau marmont in west hollywood oct... |
| negative | neutral | miyagi just got banned from yoga he was caught him sniffing ... |
| positive | neutral | hey friends dst ends sunday 11 4 just giving you a heads up ... |
| neutral | positive | them seniors gone beat the juniors tomorrow... |
| positive | neutral | user haha i meant what do you think if i m taking kl live fo... |


### nb
**Error rate:** 1477 / 3531 (41.8%)

| True | Predicted | Count |
|------|-----------|-------|
| neutral | positive | 500 |
| positive | neutral | 456 |
| negative | neutral | 263 |
| negative | positive | 101 |
| positive | negative | 81 |
| neutral | negative | 76 |

**Sample errors:**

| True | Predicted | Tweet |
|------|-----------|-------|
| neutral | positive | them seniors gone beat the juniors tomorrow... |
| positive | neutral | user haha i meant what do you think if i m taking kl live fo... |
| neutral | positive | make sure you tune into the saga exclusively on temple s onl... |
| neutral | positive | bryce harper triples home solano with the tying run in the 1... |
| positive | negative | some good stuff going down in gregory gym 2nd place tasted t... |


### glove-lstm
**Error rate:** 1219 / 3531 (34.5%)

| True | Predicted | Count |
|------|-----------|-------|
| positive | neutral | 600 |
| negative | neutral | 306 |
| neutral | positive | 182 |
| negative | positive | 58 |
| neutral | negative | 39 |
| positive | negative | 34 |

**Sample errors:**

| True | Predicted | Tweet |
|------|-----------|-------|
| negative | positive | miyagi just got banned from yoga he was caught him sniffing ... |
| positive | neutral | hey friends dst ends sunday 11 4 just giving you a heads up ... |
| neutral | positive | 13 april 1996 history is made as the metrostars and the los ... |
| neutral | positive | boom 1st post bilbao night half marathon 9km i just finished... |
| positive | neutral | saturday nov 17th 2012 q bar and grill is the place to be uf... |


## Step 12: Confusion Matrices

Rows = predicted, Columns = actual.  
Diagonal values show correct classification rates.

In [17]:
from IPython.display import display, Markdown

def confusion(id_preds, testset, classifier):
    id_gts = read_test(testset)
    gts = ['positive', 'negative', 'neutral']
    conf = {c1: {c2: 0 for c2 in gts} for c1 in gts}

    for tweetid, gt in id_gts.items():
        pred = id_preds.get(tweetid, 'neutral')
        conf[pred][gt] += 1

    table  = f'#### {classifier} — {os.path.basename(testset)}\n\n'
    table += '| Predicted ↓ / Actual → | positive | negative | neutral |\n'
    table += '|------------------------|----------|----------|---------|\n'
    for c1 in gts:
        total = sum(conf[c1].values())
        row = f'| **{c1}** |'
        for c2 in gts:
            val = conf[c1][c2] / float(total) if total > 0 else 0.0
            row += f' {val:.3f} |'
        table += row + '\n'
    table += '\n'
    display(Markdown(table))

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]

    id_preds = dict(zip(t_ids, svm_pipeline.predict(t_texts)))
    confusion(id_preds, join(DATA_DIR, ts), 'bow-svm')

    id_preds = dict(zip(t_ids, lr_pipeline.predict(t_texts)))
    confusion(id_preds, join(DATA_DIR, ts), 'wordchar-lr')

    id_preds = dict(zip(t_ids, nb_pipeline.predict(t_texts)))
    confusion(id_preds, join(DATA_DIR, ts), 'nb')

    id_preds = dict(zip(t_ids, predict_lstm(lstm_model, t_texts, word2idx, device)))
    confusion(id_preds, join(DATA_DIR, ts), 'glove-lstm')


#### bow-svm — twitter-test1.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.721 | 0.048 | 0.230 |
| **negative** | 0.166 | 0.629 | 0.206 |
| **neutral** | 0.238 | 0.128 | 0.634 |



#### wordchar-lr — twitter-test1.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.730 | 0.034 | 0.237 |
| **negative** | 0.144 | 0.656 | 0.200 |
| **neutral** | 0.229 | 0.125 | 0.646 |



#### nb — twitter-test1.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.608 | 0.066 | 0.326 |
| **negative** | 0.231 | 0.551 | 0.217 |
| **neutral** | 0.277 | 0.160 | 0.563 |



#### glove-lstm — twitter-test1.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.777 | 0.054 | 0.169 |
| **negative** | 0.128 | 0.726 | 0.147 |
| **neutral** | 0.274 | 0.140 | 0.586 |



#### bow-svm — twitter-test2.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.753 | 0.044 | 0.203 |
| **negative** | 0.147 | 0.657 | 0.196 |
| **neutral** | 0.328 | 0.087 | 0.585 |



...

#### glove-lstm — twitter-test2.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.811 | 0.058 | 0.130 |
| **negative** | 0.100 | 0.733 | 0.167 |
| **neutral** | 0.366 | 0.091 | 0.543 |



#### bow-svm — twitter-test3.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.738 | 0.047 | 0.215 |
| **negative** | 0.183 | 0.530 | 0.287 |
| **neutral** | 0.303 | 0.119 | 0.578 |



#### wordchar-lr — twitter-test3.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.747 | 0.041 | 0.212 |
| **negative** | 0.186 | 0.539 | 0.275 |
| **neutral** | 0.294 | 0.096 | 0.610 |



#### nb — twitter-test3.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.664 | 0.067 | 0.270 |
| **negative** | 0.233 | 0.462 | 0.306 |
| **neutral** | 0.305 | 0.144 | 0.551 |



#### glove-lstm — twitter-test3.txt

| Predicted ↓ / Actual → | positive | negative | neutral |
|------------------------|----------|----------|---------|
| **positive** | 0.769 | 0.059 | 0.171 |
| **negative** | 0.148 | 0.602 | 0.250 |
| **neutral** | 0.309 | 0.132 | 0.559 |



## Step 13: Classifier 5 — BERT Fine-tuning

**Model:** `bert-base-uncased` (pre-trained on BookCorpus + English Wikipedia)  
**Approach:** Fine-tune the last layer only for 3-class sentiment classification

| Component | Details |
|-----------|---------|
| Base model | bert-base-uncased |
| Max sequence length | 128 tokens |
| Batch size | 16 |
| Epochs | 3 |
| Optimiser | AdamW |
| Learning rate | 2e-5 |

### Step 13a: Load BERT Tokenizer and Model

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
import random

MODEL_NAME = 'distilbert-base-uncased'

print(f'Loading {MODEL_NAME}...')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
bert_model = bert_model.to(device)
print(f'Model loaded.')
print(f'Parameters: {sum(p.numel() for p in bert_model.parameters()):,}')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading distilbert-base-uncased...


Loading weights: 100%|██████████████████████| 100/100 [00:00<00:00, 9919.83it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Parameters: 66,955,779


### Step 13b: Prepare BERT Dataset

In [20]:
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor([LABEL_MAP[l] for l in labels], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids'     : self.encodings['input_ids'][i],
            'attention_mask': self.encodings['attention_mask'][i],
            'labels'        : self.labels[i]
        }

# 10,000 samples
random.seed(42)
indices = random.sample(range(len(all_train_texts)), 10000)
sample_texts  = [all_train_texts[i] for i in indices]
sample_labels = [all_train_labels[i] for i in indices]

print('Tokenizing...')
bert_train_dataset = BERTDataset(sample_texts, sample_labels, tokenizer)
bert_train_loader  = DataLoader(bert_train_dataset, batch_size=16, shuffle=True)

print(f'Training samples : {len(bert_train_dataset):,}')
print(f'Batches per epoch: {len(bert_train_loader):,}')

Tokenizing...
Training samples : 10,000
Batches per epoch: 625


### Step 13c: Fine-tune BERT

In [22]:
BERT_EPOCHS    = 5
optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5)

for epoch in range(1, BERT_EPOCHS + 1):
    bert_model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for i, batch in enumerate(bert_train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer_bert.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
        optimizer_bert.step()

        total_loss += loss.item() * len(labels)
        correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total      += len(labels)

        if (i + 1) % 100 == 0:
            print(f'  Batch {i+1}/{len(bert_train_loader)} | Loss: {total_loss/total:.4f} | Acc: {correct/total:.4f}')

    avg_loss  = total_loss / total
    train_acc = correct / total
    print(f'Epoch {epoch}/{BERT_EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}')

print('\nTraining complete.')

  Batch 100/625 | Loss: 0.9580 | Acc: 0.5150
  Batch 200/625 | Loss: 0.8692 | Acc: 0.5794
  Batch 300/625 | Loss: 0.8439 | Acc: 0.5927
  Batch 400/625 | Loss: 0.8199 | Acc: 0.6077
  Batch 500/625 | Loss: 0.7994 | Acc: 0.6186
  Batch 600/625 | Loss: 0.7860 | Acc: 0.6268
Epoch 1/5 | Loss: 0.7819 | Train Acc: 0.6291
  Batch 100/625 | Loss: 0.5944 | Acc: 0.7450
  Batch 200/625 | Loss: 0.5731 | Acc: 0.7581
  Batch 300/625 | Loss: 0.5703 | Acc: 0.7562
  Batch 400/625 | Loss: 0.5737 | Acc: 0.7512
  Batch 500/625 | Loss: 0.5750 | Acc: 0.7501
  Batch 600/625 | Loss: 0.5713 | Acc: 0.7530
Epoch 2/5 | Loss: 0.5700 | Train Acc: 0.7541
  Batch 100/625 | Loss: 0.3970 | Acc: 0.8406
  Batch 200/625 | Loss: 0.3766 | Acc: 0.8509
  Batch 300/625 | Loss: 0.3809 | Acc: 0.8517
  Batch 400/625 | Loss: 0.3846 | Acc: 0.8517
  Batch 500/625 | Loss: 0.3832 | Acc: 0.8535
  Batch 600/625 | Loss: 0.3839 | Acc: 0.8530
Epoch 3/5 | Loss: 0.3824 | Train Acc: 0.8537
  Batch 100/625 | Loss: 0.2410 | Acc: 0.9144
  Batch 20

### Step 13d: Evaluate DistilBERT

In [23]:
def predict_bert(model, texts, tokenizer, device, batch_size=32):
    model.eval()
    all_preds = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encodings   = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors='pt'
        )
        input_ids      = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend([IDX2LABEL[int(p)] for p in preds])

    return all_preds


for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    preds    = predict_bert(bert_model, t_texts, tokenizer, device)
    id_preds = dict(zip(t_ids, preds))
    evaluate(id_preds, join(DATA_DIR, ts), 'distilbert')

twitter-test1.txt (distilbert): 0.652
twitter-test2.txt (distilbert): 0.660
twitter-test3.txt (distilbert): 0.637


# All result

In [24]:
from IPython.display import display, Markdown


all_results = {}
classifiers = {
    'SVM'        : lambda texts: svm_pipeline.predict(texts),
    'LR'         : lambda texts: lr_pipeline.predict(texts),
    'Naive Bayes': lambda texts: nb_pipeline.predict(texts),
    'BiLSTM'     : lambda texts: predict_lstm(lstm_model, texts, word2idx, device),
    'DistilBERT' : lambda texts: predict_bert(bert_model, texts, tokenizer, device),
}

for ts in testsets:
    t_ids, t_labels, t_texts = test_data[ts]
    all_results[ts] = {}
    for clf_name, predict_fn in classifiers.items():
        preds    = predict_fn(t_texts)
        id_preds = dict(zip(t_ids, preds))
        f1 = evaluate(id_preds, join(DATA_DIR, ts), clf_name)
        all_results[ts][clf_name] = f1


table  = '## Results Summary\n\n'
table += '| Classifier | Test 1 | Test 2 | Test 3 | Average |\n'
table += '|------------|--------|--------|--------|---------|\n'

for clf_name in classifiers:
    t1  = all_results['twitter-test1.txt'][clf_name]
    t2  = all_results['twitter-test2.txt'][clf_name]
    t3  = all_results['twitter-test3.txt'][clf_name]
    avg = np.mean([t1, t2, t3])

    # highest
    best_avg = max(
        np.mean([all_results[ts][c] for ts in testsets])
        for c in classifiers
    )
    name = f'{clf_name}' if abs(avg - best_avg) < 0.001 else clf_name
    table += f'| {name} | {t1:.3f} | {t2:.3f} | {t3:.3f} | {avg:.3f} |\n'

display(Markdown(table))

twitter-test1.txt (SVM): 0.621
twitter-test1.txt (LR): 0.652
twitter-test1.txt (Naive Bayes): 0.523
twitter-test1.txt (BiLSTM): 0.563
twitter-test1.txt (DistilBERT): 0.652
twitter-test2.txt (SVM): 0.641
twitter-test2.txt (LR): 0.667
twitter-test2.txt (Naive Bayes): 0.577
twitter-test2.txt (BiLSTM): 0.574
twitter-test2.txt (DistilBERT): 0.660
twitter-test3.txt (SVM): 0.580
twitter-test3.txt (LR): 0.619
twitter-test3.txt (Naive Bayes): 0.516
twitter-test3.txt (BiLSTM): 0.542
twitter-test3.txt (DistilBERT): 0.637


## Results Summary

| Classifier | Test 1 | Test 2 | Test 3 | Average |
|------------|--------|--------|--------|---------|
| SVM | 0.621 | 0.641 | 0.580 | 0.614 |
| LR | 0.652 | 0.667 | 0.619 | 0.646 |
| Naive Bayes | 0.523 | 0.577 | 0.516 | 0.539 |
| BiLSTM | 0.563 | 0.574 | 0.542 | 0.559 |
| DistilBERT | 0.652 | 0.660 | 0.637 | 0.649 |
